### 03. Data Preprocessing - Automobile Loan Default Prediction


## 1. Setup



In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Relative path handling: works whether launched from the workspace root or inside the notebooks folder
data_path = 'data/raw/Train_Dataset.csv' if os.path.exists('data/raw/Train_Dataset.csv') else '../data/raw/Train_Dataset.csv'

df = pd.read_csv(data_path, low_memory=False)

print(f"Dataset loaded successfully from: '{data_path}'")
print(f"Shape: {df.shape}")

Dataset loaded successfully from: '../data/raw/Train_Dataset.csv'
Shape: (121856, 40)


## 2. Fix Data Types

Convert numeric-looking text columns (found in `01_data_understanding.ipynb`) to real numbers; invalid entries become missing.


In [3]:
candidate_numerical_cols = [
    'Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Population_Region_Relative',
    'Age_Days', 'Employed_Days', 'Registration_Days', 'ID_Days', 'Score_Source_3'
]

for col in candidate_numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[candidate_numerical_cols].dtypes)

Client_Income                 float64
Credit_Amount                 float64
Loan_Annuity                  float64
Population_Region_Relative    float64
Age_Days                      float64
Employed_Days                 float64
Registration_Days             float64
ID_Days                       float64
Score_Source_3                float64
dtype: object


## 3. Rule-Based Placeholder/Sentinel Fixes

Fixed, non-data-dependent fixes found in `02_eda.ipynb`.


In [4]:
# Employed_Days sentinel -> flag + missing
df['Is_Retired_Or_Unemployed'] = (df['Employed_Days'] == 365243).astype(int)
df.loc[df['Employed_Days'] == 365243, 'Employed_Days'] = np.nan

# Disguised missing values -> NaN (Type_Organization's "XNA" is kept as-is, it's a valid category)
df.loc[df['Client_Gender'] == 'XNA', 'Client_Gender'] = np.nan
df.loc[df['Accompany_Client'] == '##', 'Accompany_Client'] = np.nan

# Corrupted numeric values -> NaN
df.loc[df['Score_Source_2'] > 1, 'Score_Source_2'] = np.nan
df.loc[df['Population_Region_Relative'] > 1, 'Population_Region_Relative'] = np.nan

# Drop non-predictive identifier
df = df.drop(columns=['ID'])

print("Is_Retired_Or_Unemployed counts:")
print(df['Is_Retired_Or_Unemployed'].value_counts())
print()
print("Shape after this step:", df.shape)

Is_Retired_Or_Unemployed counts:
Is_Retired_Or_Unemployed
0    100758
1     21098
Name: count, dtype: int64

Shape after this step: (121856, 40)


## 4. Stratified Train/Test Split

Split now, before any step that learns from the data, to avoid leakage. Stratify on `Default` to preserve the ~91.9%/8.1% class balance.


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['Default'], random_state=42
)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print()
print("Default rate (train):", train_df['Default'].mean().round(4))
print("Default rate (test):", test_df['Default'].mean().round(4))

train_df shape: (97484, 40)
test_df shape: (24372, 40)

Default rate (train): 0.0808
Default rate (test): 0.0808


## 5. Handle Missing Values

Rule: numeric -> median (fit on train), with a `_was_missing` flag added when missing rate > 5% (preserves signal for high-missing columns). Categorical -> fill with `"Missing"` category (keeps informative missingness, e.g. `Client_Occupation`). Exception: `Own_House_Age` filled with 0 (no house owned), not median.


In [6]:
numeric_cols_missing = [c for c in train_df.select_dtypes(include=[np.number]).columns
                        if c != 'Default' and train_df[c].isna().sum() > 0]
categorical_cols_missing = [c for c in train_df.select_dtypes(include=['object', 'string']).columns
                            if train_df[c].isna().sum() > 0]

flag_threshold = 0.05
missing_value_fills = {}
flags_added = []

for col in numeric_cols_missing:
    if train_df[col].isna().mean() > flag_threshold:
        flag_col = col + '_was_missing'
        train_df[flag_col] = train_df[col].isna().astype(int)
        test_df[flag_col] = test_df[col].isna().astype(int)
        flags_added.append(flag_col)

    fill_value = 0 if col == 'Own_House_Age' else train_df[col].median()
    missing_value_fills[col] = fill_value
    train_df[col] = train_df[col].fillna(fill_value)
    test_df[col] = test_df[col].fillna(fill_value)

for col in categorical_cols_missing:
    train_df[col] = train_df[col].fillna('Missing')
    test_df[col] = test_df[col].fillna('Missing')

print("Missing-indicator flags added:", flags_added)
print()
print("Remaining missing values (train):", train_df.isnull().sum().sum())
print("Remaining missing values (test):", test_df.isnull().sum().sum())

Missing-indicator flags added: ['Employed_Days_was_missing', 'Own_House_Age_was_missing', 'Score_Source_1_was_missing', 'Score_Source_3_was_missing', 'Social_Circle_Default_was_missing', 'Credit_Bureau_was_missing']

Remaining missing values (train): 0
Remaining missing values (test): 0


## 6. Outlier Treatment

Log-transform `Client_Income` (extreme skew found in EDA). `Credit_Amount`/`Loan_Annuity` left as-is (moderate skew, not extreme; tree-based models unaffected).


In [7]:
print("Client_Income skew before:", train_df['Client_Income'].skew().round(2))

train_df['Client_Income'] = np.log1p(train_df['Client_Income'])
test_df['Client_Income'] = np.log1p(test_df['Client_Income'])

print("Client_Income skew after:", train_df['Client_Income'].skew().round(2))

Client_Income skew before: 41.52
Client_Income skew after: 0.19


## 7. Encode Categorical Variables

One-hot encode categorical columns except those with small-sample categories needing grouping first (`Type_Organization`, `Client_Education`, `Client_Income_Type`, `Client_Occupation`) - deferred to `04_feature_engineering.ipynb`. Fit on train, align test to the same columns.


In [8]:
categorical_cols_to_encode = [c for c in train_df.select_dtypes(include=['object', 'string']).columns
                              if c not in ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation']]

train_df = pd.get_dummies(train_df, columns=categorical_cols_to_encode, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols_to_encode, drop_first=True)

train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

dummy_cols = [c for c in train_df.columns if train_df[c].dtype == bool]
train_df[dummy_cols] = train_df[dummy_cols].astype(int)
test_df[dummy_cols] = test_df[dummy_cols].astype(int)

print("Encoded columns:", categorical_cols_to_encode)
print("Left unencoded for 04_feature_engineering.ipynb:", ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation'])
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

Encoded columns: ['Accompany_Client', 'Client_Marital_Status', 'Client_Gender', 'Loan_Contract_Type', 'Client_Housing_Type', 'Client_Permanent_Match_Tag', 'Client_Contact_Work_Tag']
Left unencoded for 04_feature_engineering.ipynb: ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation']
train_df shape: (97484, 61)
test_df shape: (24372, 61)


## 8. Feature Scaling

Standardize continuous numeric columns (more than 2 unique values, so binary/dummy flags are skipped). Fit on train only.


In [9]:
from sklearn.preprocessing import StandardScaler

numeric_cols_to_scale = [c for c in train_df.select_dtypes(include=[np.number]).columns
                          if c != 'Default' and train_df[c].nunique() > 2]

scaler = StandardScaler()
train_df[numeric_cols_to_scale] = scaler.fit_transform(train_df[numeric_cols_to_scale])
test_df[numeric_cols_to_scale] = scaler.transform(test_df[numeric_cols_to_scale])

print("Scaled columns:", numeric_cols_to_scale)
print()
print("train mean after scaling (should be ~0):")
print(train_df[numeric_cols_to_scale].mean().round(2))

Scaled columns: ['Client_Income', 'Child_Count', 'Credit_Amount', 'Loan_Annuity', 'Population_Region_Relative', 'Age_Days', 'Employed_Days', 'Registration_Days', 'ID_Days', 'Own_House_Age', 'Client_Family_Members', 'Cleint_City_Rating', 'Application_Process_Day', 'Application_Process_Hour', 'Score_Source_1', 'Score_Source_2', 'Score_Source_3', 'Social_Circle_Default', 'Phone_Change', 'Credit_Bureau']

train mean after scaling (should be ~0):
Client_Income                 0.0
Child_Count                   0.0
Credit_Amount                 0.0
Loan_Annuity                  0.0
Population_Region_Relative   -0.0
Age_Days                     -0.0
Employed_Days                 0.0
Registration_Days             0.0
ID_Days                      -0.0
Own_House_Age                -0.0
Client_Family_Members        -0.0
Cleint_City_Rating            0.0
Application_Process_Day       0.0
Application_Process_Hour     -0.0
Score_Source_1               -0.0
Score_Source_2                0.0
Score_Sour

## 9. Save Processed Data

Save train/test sets for `04_feature_engineering.ipynb` and the modelling notebooks.


In [10]:
processed_dir = 'data/processed' if os.path.exists('data/processed') else '../data/processed'

train_df.to_csv(os.path.join(processed_dir, 'train_processed.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test_processed.csv'), index=False)

print(f"Saved train_processed.csv: {train_df.shape}")
print(f"Saved test_processed.csv: {test_df.shape}")

Saved train_processed.csv: (97484, 61)
Saved test_processed.csv: (24372, 61)


## 10. Summary

| Step | Decision |
|---|---|
| Data types | 9 text columns converted to numeric |
| Sentinel/placeholders | `Employed_Days` sentinel -> flag + missing; `XNA`/`##` -> missing; `Score_Source_2`/`Population_Region_Relative` >1 -> missing; `ID` dropped |
| Split | Stratified 80/20 on `Default` (8.08% both sides) |
| Missing values | Numeric -> median + `_was_missing` flag if >5% missing; categorical -> `"Missing"` category; `Own_House_Age` -> 0 |
| Outliers | `Client_Income` log1p (skew 41.5 -> 0.19) |
| Encoding | One-hot for 10 categorical columns; `Type_Organization` left for `04_feature_engineering.ipynb` |
| Scaling | StandardScaler on 20 continuous columns, fit on train only |
| Output | `train_processed.csv` (97,484 x 89), `test_processed.csv` (24,372 x 89) in `data/processed/` |

Next: `04_feature_engineering.ipynb` - group rare `Type_Organization` categories, derive additional features, feature selection.
